# Day 01 · Setup and your first retail DataFrame

**Priority version:** 11 tasks that cover every unique concept of the day. Tasks 1–10 are the core concepts, Task 11 bundles the smaller items into quick mini-problems, and the last cell answers the conceptual questions.

Each task has a markdown cell (problem + expected result) followed by its solution cell.

Data: `/Volumes/dbw_pyspark_course/retail/raw/` (6 CSVs + `manifest.json`)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json

RAW = "/Volumes/dbw_pyspark_course/retail/raw"

### Task 1 · Configure the session
**Covers:** A2, Q3, Q20

**Problem:** The course runs Spark with `local[2]`, UTC timestamps and 4 shuffle partitions. On Databricks the session (`spark`) already exists, and the cluster decides the master. Set the two runtime settings, check them, and see what master this cluster uses.

**Expected:** timezone `UTC`, shuffle partitions `4`, master like `local[*, 4]` (all cores on one VM, failed tasks retried up to 4 times). The course's `local[2]` would mean one machine with 2 threads.

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")   # default is 200: far too many tiny tasks for 300 rows

print("timezone          :", spark.conf.get("spark.sql.session.timeZone"))
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("master            :", spark.sparkContext.master)

### Task 2 · Read orders and inspect it
**Covers:** A3, Q11, Q12

**Problem:** Read `orders.csv` using its header row. Print the schema, count the rows and compare the count with `manifest.json`, then show 10 rows. Which columns were read as strings?

**Expected:** 300 rows (same as the manifest). All 6 columns are `string`, because a CSV carries no types and we did not give Spark a schema. `order_ts` being a string means no date maths yet; Day 02 fixes this.

In [0]:
orders = spark.read.option("header", True).csv(f"{RAW}/orders.csv")
orders.printSchema()

manifest = json.loads(open(f"{RAW}/manifest.json").read())
print("count:", orders.count(), "| manifest:", manifest["raw_counts"]["orders"])
print("string columns:", [f.name for f in orders.schema.fields if isinstance(f.dataType, T.StringType)])

orders.show(10, truncate=False)

### Task 3 · Transformations vs actions (lazy evaluation)
**Covers:** A4, Q7, Q8, Q13

**Problem:** Select `order_id`, `customer_id` and `status`, and filter to completed orders. Find out which line actually makes Spark read the data.

**Expected:**
- Cell 1 (`select`, `filter`, `explain`): **no "Spark Jobs" line** under the cell. It only builds a plan. `explain()` shows `ReadSchema` with only the 3 selected columns: Spark prunes columns because it can see the whole plan before running anything.
- Cell 2 (`show`, `count`): **"Spark Jobs" appears**. Actions run the plan. The count is 270.

In [0]:
selected = orders.select("order_id", "customer_id", "status")   # transformation: no data read
completed = selected.filter(F.col("status") == "completed")      # transformation: no data read
completed.explain()                                              # shows the plan only

In [0]:
selected.show(10, truncate=False)                                # action -> job runs
print("completed orders:", completed.count())                    # action -> job runs

### Task 4 · Same action twice = same work twice
**Covers:** Q21

**Problem:** Count the same filtered DataFrame twice. How many times does Spark read the file?

**Expected:** twice. A DataFrame is only a plan, not stored results, so every action re-runs the whole plan from the CSV. Expand "Spark Jobs" under the cell: there is a set of jobs for each count. Caching (Day 17) is how you avoid this.

In [0]:
cancelled = orders.filter(F.col("status") == "cancelled")
print("first count :", cancelled.count())
print("second count:", cancelled.count())

### Task 5 · Build a DataFrame with an explicit schema
**Covers:** A5, Q6, Q16

**Problem:** Create a five-row customer DataFrame by hand with an explicit schema where every column is a string. Compare it with letting Spark infer the types.

**Expected:** the schema is exactly what we declared (`customer_id` not nullable, the rest nullable). Without a schema, Spark guesses: a Python `int` becomes `bigint` (long). A schema describes column **names, types and nullability**.

In [0]:
customer_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), nullable=False),
    T.StructField("name",        T.StringType(), nullable=True),
    T.StructField("email",       T.StringType(), nullable=True),
    T.StructField("city",        T.StringType(), nullable=True),
    T.StructField("signup_date", T.StringType(), nullable=True),
])

customers_fx = spark.createDataFrame([
    ("C001", "Asha",  "asha@example.test",  "Delhi",     "2026-06-01"),
    ("C002", "Ravi",  "",                   " mumbai ",  "2026-06-02"),   # blank email, messy city (like the real data)
    ("C003", "Meena", "meena@example.test", "Bengaluru", "2026-06-03"),
    ("C004", "Kiran", None,                 "CHENNAI",   "2026-06-04"),   # null email
    ("C005", "Dev",   "dev@example.test",   "Pune",      "2026-06-05"),
], customer_schema)

customers_fx.printSchema()
customers_fx.show(truncate=False)

# Compare: no schema -> Spark infers types
customer_new_schema = spark.createDataFrame([("O1", 2)], ["order_id", "quantity"])
customer_new_schema.printSchema()

### Task 6 · Counting without Python loops
**Covers:** Q15, E2, Q18

**Problem:**
1. Count completed and cancelled orders in **one pass**, with no Python loop.
2. Count orders by channel.
3. List the distinct channel values. Why is that result small?

**Expected:** completed 270, cancelled 30 · store 200, online 100 · 2 distinct channels. `distinct` returns one row per unique value, and `channel` has only 2 values, so the result stays tiny however many orders exist.

In [0]:
# 1. Conditional aggregation: F.when gives null when the condition is false, and count() skips nulls
orders.agg(
    F.count(F.when(F.col("status") == "completed", True)).alias("completed"),
    F.count(F.when(F.col("status") == "cancelled", True)).alias("cancelled"),
).show()

# 2. Group and count
orders.groupBy("channel").count().show()

# 3. Distinct values
orders.select("channel").distinct().show()

### Task 7 · show vs take vs collect
**Covers:** E3, Q9, Q19

**Problem:** On the five-row customer DataFrame, compare `show(3)`, `take(3)` and `collect()`. What does each one return, and why is `collect()` dangerous on big data?

**Expected:**
- `show(3)` prints 3 rows and returns `None`.
- `take(3)` returns a Python list of 3 `Row` objects.
- `collect()` returns **every** row to the driver. On a billion-row table that is a billion rows in one machine's memory, which crashes the driver. To look at data safely, use `show(5)` or `limit(5)`.

In [0]:
shown = customers_fx.show(3)
taken = customers_fx.take(3)
collected = customers_fx.collect()

print("show(3)   returned:", shown)
print("take(3)   returned:", len(taken), "rows ->", taken[0])
print("collect() returned:", len(collected), "rows")
print("a Row works like a record:", taken[0].customer_id, taken[0]["city"])

### Task 8 · DataFrames are immutable
**Covers:** Q17, Q25

**Problem:** Rename `status` to `order_status`, and add a new column `is_cancelled`. Show that the original `orders` DataFrame is unchanged.

**Expected:** `orders` still has 6 columns, including `status`. Each call returns a **new** DataFrame, so you must assign the result to a variable.

In [0]:
renamed = orders.withColumnRenamed("status", "order_status")
flagged = orders.withColumn("is_cancelled", F.col("status") == "cancelled")

print("orders  :", orders.columns)
print("renamed :", renamed.columns)
print("flagged :", flagged.columns)
print("same object?", flagged is orders)

### Task 9 · Read a Spark error message
**Covers:** Q23

**Problem:** Select a column with a typo (`order_idd`) and read the error.

**Expected:** `[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column ... with name 'order_idd' cannot be resolved. Did you mean one of the following? ['order_id', ...]`. Spark checks column names against the schema while building the plan, so the error appears at `select()`, before any data is read.

In [0]:
try:
    orders.select("order_idd").show()
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:300])

### Task 10 · Row order is not guaranteed
**Covers:** Q24

**Problem:** Take the "first 5" orders after spreading the data across 4 partitions, then again after sorting.

**Expected:** without `orderBy`, the first 5 are not O0001–O0005. Row order depends on partitions and task timing. Only `orderBy` guarantees order. Sort only when you need it, because sorting costs a shuffle.

In [0]:
print("no orderBy  :", [r.order_id for r in orders.repartition(4).take(5)])
print("with orderBy:", [r.order_id for r in orders.repartition(4).orderBy("order_id").take(5)])

### Task 11 · Quick extras (six mini-problems)
**Covers:** E1, Q14, E4, E5, Q29, Q30, Q26

| | Problem | Expected |
|---|---|---|
| a | Filter orders from store S01 and show only their IDs (E1, Q14) | 67 orders |
| b | Change an assumption: what if `status` arrives as `Completed ` or `COMPLETED`? (E4) | exact match finds 1, trimmed + lower-cased finds 3. An exact match silently undercounts |
| c | Write a check that fails when the data is broken: `order_id` must be unique and non-null (E5) | real orders `True`, copy with a duplicate added `False` |
| d | When does a missing input path fail: at `read()` or at the action? (Q29) | at `read()`, with `PATH_NOT_FOUND`. Spark lists the files while building the DataFrame |
| e | Write a reusable session factory and a five-row smoke check (Q30) | `True` |
| f | Inspect a huge table safely (Q26) | only small results come back: 5 rows, counts from a sample, one row of min/max |

In [0]:
# (a) E1 / Q14 · orders from store S01, IDs only
s01 = orders.filter(F.col("store_id") == "S01").select("order_id")
s01.show(5)
print("(a) S01 orders:", s01.count())

# (b) E4 · change an assumption: status may not be clean lowercase
messy = spark.createDataFrame(
    [("X1", "completed"), ("X2", "Completed "), ("X3", "cancelled"), ("X4", "COMPLETED")],
    "order_id string, status string")
print("(b) exact match:", messy.filter(F.col("status") == "completed").count(),
      "| trimmed + lower-cased:", messy.filter(F.lower(F.trim("status")) == "completed").count())

# (c) E5 · edge-case check: order_id must be unique and not null
def is_unique_key(df, key):
    stats = df.agg(F.count(F.lit(1)).alias("rows"),
                   F.countDistinct(key).alias("distinct"),
                   F.sum(F.col(key).isNull().cast("int")).alias("nulls")).first()
    return stats["nulls"] == 0 and stats["rows"] == stats["distinct"]

broken = orders.unionByName(orders.limit(1))   # deliberately add one duplicate order
print("(c) real orders unique:", is_unique_key(orders, "order_id"),
      "| broken copy unique:", is_unique_key(broken, "order_id"))

# (d) Q29 · missing path: which step fails?
step = "read()"
try:
    missing = spark.read.option("header", True).csv(f"{RAW}/no_such_file.csv")
    step = "count()"
    missing.count()
except Exception as e:
    print(f"(d) failed at {step}:", str(e).splitlines()[0][:150])

# (e) Q30 · reusable session factory + five-row smoke check
from pyspark.sql import SparkSession

def get_spark(app_name="RetailAssignment"):
    """Reuse the active session (Databricks), or build the course's local one."""
    session = SparkSession.getActiveSession() or (
        SparkSession.builder.master("local[2]").appName(app_name).getOrCreate())
    session.conf.set("spark.sql.session.timeZone", "UTC")
    session.conf.set("spark.sql.shuffle.partitions", "4")
    return session

def smoke_check(session):
    df = session.createDataFrame([(i,) for i in range(1, 6)], "n int")
    return df.count() == 5 and df.agg(F.sum("n")).first()[0] == 15

print("(e) smoke check passed:", smoke_check(get_spark()))

# (f) Q26 · safe inspection: bounded rows, a seeded sample, and aggregates. Never collect() the full table.
orders.limit(5).show()
orders.sample(fraction=0.1, seed=42).groupBy("status").count().show()
orders.agg(F.min("order_ts"), F.max("order_ts"), F.countDistinct("store_id")).show()

### Concept notes (Q1, Q4, Q5, Q10, Q22, Q27, Q28): read once, then rewrite in your own words
- **Why Spark (Q1):** when data is too big for one machine, Spark splits it into **partitions** and processes them in parallel on many machines. If a machine fails, Spark recomputes the lost part from the plan.
- **Driver (Q4):** runs your notebook code, builds and optimises the plan, splits it into jobs → stages → tasks, sends the tasks to executors, and receives the results of actions.
- **Executors (Q5):** worker processes that run the tasks on their partitions (read, filter, aggregate, shuffle) and hold cached data.
- **Where things live (Q27):** the `spark` session and your DataFrame variables (which are just plans) live on the **driver**. The actual rows live in partitions on the **executors**, and only reach the driver through `show`, `take` or `collect`.
- **Stopping the session (Q10):** locally you call `spark.stop()`. On Databricks you don't, because the cluster owns the session; terminate the cluster instead.
- **300 rows proves nothing about scale (Q22):** the data fits in one machine's memory, so there is no network shuffle, no skew and no machine failures, and startup overhead dominates the run time. Scale is shown with data far larger than one machine, spread across many executors.
- **Driver vs distributed in this notebook (Q28):** on the driver: Python variables, `print`, functions, `if`, and the results of `take`/`collect`. Distributed: the reading, filtering, counting and grouping that run as tasks on executors when an action is called.